In [1]:
import pandas as pd
import numpy as np
import math
import catboost
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier


## Метрика

In [2]:
def ndcg_at_k(rel, k=5):
    """DCG@k для бинарной релевантности (0/1)"""
    top_k = rel[:k]
    dcg = sum(1.0 / math.log2(i+2) for i, g in enumerate(top_k) if g == 1)

    ideal_rel = sorted(rel, reverse=True)[:k]
    idcg = sum(1.0 / math.log2(i+2) for i, g in enumerate(ideal_rel) if g == 1)

    return dcg / idcg if idcg > 0 else np.nan

def mean_ndcg_at_5(df, request_col='request_id', score_col='score', target_col='is_deal'):
    """
    Вычисляет средний NDCG@5 по всем request_id в датафрейме.

    Параметры
    ----------
    df : pd.DataFrame
    Колонки: request_id, deal, score, target (0/1).
    Для каждого request_id может быть от 1 до 50 строк (предложений),
    и ровно одна строка с target=1 (покупка).
    request_col : str
    score_col : str
    target_col : str

    Возврат
    -------
    float : средний NDCG@5
    """
    # Сортируем внутри каждого request_id по убыванию скоров модели
    df_sorted = df.sort_values([request_col, score_col], ascending=[True, False])

    # Группируем и считаем NDCG@5
    ndcg_per_request = df_sorted.groupby(request_col)[target_col].apply(
    lambda x: ndcg_at_k(x.tolist(), k=5)
    )
    return np.nanmean(ndcg_per_request)

# Выгрузка выборки

In [3]:
df_train = pd.read_parquet('train_dataset_small.pq')
df_features = pd.read_parquet('features_small.pq')
df_test = pd.read_parquet('test_dataset_small.pq')

# Пример формирования выборки

In [4]:
df_train = pd.merge(df_train, df_features, on=['app_id', 'date_part'])
df_test = pd.merge(df_test, df_features, on=['app_id', 'date_part'])

In [6]:
target = 'is_deal'


columns_to_drop = ['app_id', 'request_received', 'date_part', target]
features = ['req_loan_amount', 'req_term', 'limit', 'term']+['age']

request_id_train, request_id_val = train_test_split(df_train.request_id.unique(), test_size=0.2, random_state=42)
train = df_train[df_train.request_id.isin(request_id_train)]
val = df_train[df_train.request_id.isin(request_id_val)]

X_train = train[features].fillna(0)
X_val = val[features].fillna(0)

y_train = train[target]
y_val = val[target]




# Пример обучения модели

In [8]:
rf_model = RandomForestClassifier(n_estimators=100,max_depth=5, random_state=42, n_jobs=-1)

# Train the model
print("Training RandomForestClassifier...")
rf_model.fit(X_train, y_train)
print("Training complete.")

Training RandomForestClassifier...
Training complete.


In [14]:
train.loc[:, 'score'] = rf_model.predict_proba(X_train.loc[:, :])[:, 1]
val.loc[:, 'score'] = rf_model.predict_proba(X_val.loc[:, :])[:, 1]


print(f"NDCG@5 на обучении {mean_ndcg_at_5(train)}")

print(f"NDCG@5 на валидации {mean_ndcg_at_5(val)}")

NDCG@5 на обучении 0.5740317003290595
NDCG@5 на валидации 0.5679099928172667


# Commit

In [10]:
df_test.loc[:, 'score'] = rf_model.predict_proba(df_test[features].fillna(0))[:, 1].round(6)

In [11]:
answer_cols = ['request_id', 'variant_no', 'score']
df_test[answer_cols].to_csv('commit.csv', sep=';',index=False)